In [1]:
import numpy as np

In [2]:
init_reward = np.array([[0.02, 0.01, -0.02, 0.03], [0, 0.01, 0.02, 0.01], [0.02, -0.01, 0.01, 0], [0.02, 0.01, 0.02, 0.01]])

In [3]:
NumRows = 4
NumColumns = 4
StartState = (1,0)
WinState = (2,3)
Deterministic = True

In [4]:
class State:
    def __init__(self, state=StartState):
        self.board = np.zeros([NumRows,NumColumns])
        self.state = state
        self.GameOver = False
        self.determine = Deterministic

    def GiveReward(self):
        if self.state == WinState:
            return 1
        else:
            return 0
    
    def gameEnd(self):
        if self.state == WinState:
            self.GameOver = True
        
    def NextMove(self, action):
        if self.determine == True:
            if action == "up":
                nxtState = (self.state[0]-1, self.state[1])
            elif action == "down":
                nxtState = (self.state[0]+1, self.state[1])
            elif action == "left":
                nxtState = (self.state[0], self.state[1]-1)
            else:
                nxtState = (self.state[0], self.state[1]+1)
            if (nxtState[0] >= 0) and (nxtState[0] <= 3):
                if (nxtState[1] >= 0) and (nxtState[1] <= 3):
                    return nxtState
            return self.state
        
    def showBoard(self):
        self.board[self.state] = 1
        for i in range(0, NumRows):
            print('-----------------')
            out = '| '
            for j in range(0, NumColumns):
                if self.board[i, j] == 1:
                    token = '*'
                if self.board[i, j] == -1:
                    token = 'z'
                if self.board[i, j] == 0:
                    token = '0'
                out += token + ' | '
            print(out)
        print('-----------------')  


In [5]:
s = State()
s.state

(1, 0)

In [115]:
StateLog = []
rewardLog = []

In [116]:
class Agent:
    
    def __init__(self):
        self.states = []
        self.actions = ["up", "down", "left", "right"]
        self.State = State()
        self.lr = 0.2
        self.GameOver= self.State.GameOver
        
        self.state_values = {}
        for i in range(NumRows):
            for j in range(NumColumns):
                self.state_values[(i, j)] = 0
    
    def chooseAction(self):
        action = np.random.choice(self.actions)
        return action
    
    def takeAction(self, action):
        position = self.State.NextMove(action)
        return State(state=position)     
    
    def reset(self):
        self.states = []
        self.State = State()
        self.GameOver = self.State.GameOver
    
    def play(self, rounds=10):
        i = 0
        while i < rounds:
            if self.State.GameOver:
                reward = self.State.GiveReward()
                self.state_values[self.State.state] = reward
                print("Game End Reward", reward)
                rewardLog.append(init_reward[self.State.state] + 1)
                StateLog.append(self.State.state)
                for s in reversed(self.states):
                    reward = self.state_values[s] + init_reward[s]
                    self.state_values[s] = round(reward, 3)
                self.reset()
                i += 1
            else:
                action = self.chooseAction()
                self.states.append(self.State.NextMove(action))
                print("current position {} action {} Reward {}".format(self.State.state, action, init_reward[self.State.state]))
                if self.State.state != StartState:
                    rewardLog.append(init_reward[self.State.state])
                    StateLog.append(self.State.state)
                self.State = self.takeAction(action)
                self.State.gameEnd()
                print("nxt state", self.State.state)
                print("---------------------")
                self.GameOver = self.State.GameOver
    
    def showValues(self):
        for i in range(0, NumRows):
            print('----------------------------------')
            out = '| '
            for j in range(0, NumColumns):
                out += str(self.state_values[(i, j)]) + ' | '
            print(out)
        print('----------------------------------') 

In [117]:
ag = Agent()

ag.play(1)

current position (1, 0) action left Reward 0.0
nxt state (1, 0)
---------------------
current position (1, 0) action down Reward 0.0
nxt state (2, 0)
---------------------
current position (2, 0) action up Reward 0.02
nxt state (1, 0)
---------------------
current position (1, 0) action right Reward 0.0
nxt state (1, 1)
---------------------
current position (1, 1) action right Reward 0.01
nxt state (1, 2)
---------------------
current position (1, 2) action up Reward 0.02
nxt state (0, 2)
---------------------
current position (0, 2) action down Reward -0.02
nxt state (1, 2)
---------------------
current position (1, 2) action down Reward 0.02
nxt state (2, 2)
---------------------
current position (2, 2) action down Reward 0.01
nxt state (3, 2)
---------------------
current position (3, 2) action down Reward 0.02
nxt state (3, 2)
---------------------
current position (3, 2) action up Reward 0.02
nxt state (2, 2)
---------------------
current position (2, 2) action down Reward 0.01
n

In [118]:
ag.showValues()

----------------------------------
| 0 | 0 | -0.02 | 0 | 
----------------------------------
| 0.0 | 0.02 | 0.06 | 0 | 
----------------------------------
| 0.02 | -0.03 | 0.05 | 1.0 | 
----------------------------------
| 0 | 0.03 | 0.2 | 0.02 | 
----------------------------------


In [119]:
StateLog

[(2, 0),
 (1, 1),
 (1, 2),
 (0, 2),
 (1, 2),
 (2, 2),
 (3, 2),
 (3, 2),
 (2, 2),
 (3, 2),
 (3, 2),
 (3, 1),
 (2, 1),
 (2, 2),
 (3, 2),
 (3, 2),
 (3, 1),
 (2, 1),
 (3, 1),
 (3, 2),
 (3, 2),
 (3, 3),
 (3, 2),
 (2, 2),
 (2, 1),
 (1, 1),
 (1, 2),
 (2, 2),
 (3, 2),
 (3, 3),
 (2, 3)]

In [120]:
rewardLog

[0.02,
 0.01,
 0.02,
 -0.02,
 0.02,
 0.01,
 0.02,
 0.02,
 0.01,
 0.02,
 0.02,
 0.01,
 -0.01,
 0.01,
 0.02,
 0.02,
 0.01,
 -0.01,
 0.01,
 0.02,
 0.02,
 0.01,
 0.02,
 0.01,
 -0.01,
 0.01,
 0.02,
 0.01,
 0.02,
 0.01,
 1.0]

In [124]:
print(len(StateLog),
len(rewardLog))

31 31


In [125]:
Policy = []
for i in range(len(StateLog)):
    Policy.append((StateLog[i],rewardLog[i]))
Policy

[((2, 0), 0.02),
 ((1, 1), 0.01),
 ((1, 2), 0.02),
 ((0, 2), -0.02),
 ((1, 2), 0.02),
 ((2, 2), 0.01),
 ((3, 2), 0.02),
 ((3, 2), 0.02),
 ((2, 2), 0.01),
 ((3, 2), 0.02),
 ((3, 2), 0.02),
 ((3, 1), 0.01),
 ((2, 1), -0.01),
 ((2, 2), 0.01),
 ((3, 2), 0.02),
 ((3, 2), 0.02),
 ((3, 1), 0.01),
 ((2, 1), -0.01),
 ((3, 1), 0.01),
 ((3, 2), 0.02),
 ((3, 2), 0.02),
 ((3, 3), 0.01),
 ((3, 2), 0.02),
 ((2, 2), 0.01),
 ((2, 1), -0.01),
 ((1, 1), 0.01),
 ((1, 2), 0.02),
 ((2, 2), 0.01),
 ((3, 2), 0.02),
 ((3, 3), 0.01),
 ((2, 3), 1.0)]